***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_raw  = os.path.join(path_sp, 'Process Revamp', 'Task 9. Collect new data', 'BLS')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
path_code    = os.path.join(path_git, 'Data', 'BLS')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Data', 'BLS', 'config')


## User defined functions ---

exec(open(os.path.join(path_config0,     'Functions.py')).read())
exec(open(os.path.join(path_config , 'BLS Functions.py')).read())

## Setting the API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
exec(open(os.path.join(path_config, 'api_key.txt')).read())
api_key = dict_api[user]



***

Importing

***

In [ ]:


# Set browser user agent
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0'}

url = "https://www.bls.gov/cew/classifications/areas/qcew-county-msa-csa-crosswalk.xlsx"

request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 2, engine='openpyxl')
df



In [ ]:


# Read in State Abbreviations mapping
df_states = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx')
                            , sheet_name = 'StateNames')
df_states = df_states[['State', 'Postal']]

# Extract State Abbreviation from MSA label
def extract_state(text):
    if text == 'District of Columbia':
        pass
    else:
        text = text.split(',')[1].strip()
    return text

df2 = df.copy()
df2.loc[:, 'State'] = df2['County Title'].apply(extract_state)
df2 = df2.merge(df_states, on = 'State', how = 'left')
df2.loc[(df2['Postal'].isna()) & (df2['State'] == 'District of Columbia'), 'Postal'] = 'DC'
df2.loc[:, 'County Code'] = df2['County Code'].astype(str).apply(lambda s : s[-3:])
df2.loc[:, 'MSA Code'   ] = df2['MSA Code'   ].apply(lambda s : s[1:] + '0')
df2 = df2.drop(['County Title', 'CSA Code', 'CSA Title', 'State'], axis = 1)
df2 = df2.rename(columns = {'County Code':'County FIPS', 'MSA Code':'MSA_ID', 'MSA Title':'MSA', 'Postal':'State'})
df2 = df2.sort_values('County FIPS')
df2



In [ ]:


df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
df2 = df2.drop('MSA_ID', axis=1)
df_fips = df_fips.merge(df2, on = ['State', 'County FIPS'], how = 'left')
df_fips['MSA_bls'] = df_fips['MSA']
df_fips = df_fips.drop('MSA', axis=1)
df_fips



***

Exporting

***

In [ ]:


# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_fips.to_excel(writer, index = False, sheet_name = 'CountyFIPS')

